[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/github-actions-certified/notebooks/day-03-reusable-workflows.ipynb#scrollTo=c1d2e3f4)

---
# Day 3 · Reusable Workflows and Composite Actions
**certified-journeys / github-actions-certified** · Day 3 · Reuse Patterns

> **Goal for today:** By the end of this notebook you can extract a shared lint+test job into a reusable workflow, call it from two callers, and build a composite action that caches Python dependencies — and you know exactly when to use each pattern.


In [ ]:
%pip install -q pyyaml


## Step 1 · Reusable workflows — the `workflow_call` trigger

A **reusable workflow** is a regular `.github/workflows/*.yml` file that declares `on: workflow_call` instead of (or in addition to) push/pull_request. Any other workflow can invoke it as if it were a job.

Key anatomy:

```yaml
# .github/workflows/lint-and-test.yml  (the CALLED workflow)
on:
  workflow_call:
    inputs:
      python-version:
        type: string
        required: true
    secrets:
      CODECOV_TOKEN:
        required: true
```

```yaml
# any-caller.yml  (the CALLER workflow)
jobs:
  call-lint-test:
    uses: ./.github/workflows/lint-and-test.yml
    with:
      python-version: '3.11'
    secrets:
      CODECOV_TOKEN: ${{ secrets.CODECOV_TOKEN }}
```

**Critical rule:** Only reusable workflows can receive `secrets:` — composite actions cannot. This is the main architectural reason to reach for a reusable workflow over a composite action when secret access is required.


In [ ]:
import pathlib, yaml, json, textwrap

wf_dir = pathlib.Path(".github/workflows")
wf_dir.mkdir(parents=True, exist_ok=True)

# ── The reusable workflow (called) ──────────────────────────────────────────
reusable_wf_yaml = """\
name: Lint and Test (Reusable)

on:
  workflow_call:
    inputs:
      python-version:
        description: 'Python version to use'
        type: string
        required: true
      working-directory:
        description: 'Directory containing source code'
        type: string
        required: false
        default: '.'
    secrets:
      CODECOV_TOKEN:
        required: false

jobs:
  lint-and-test:
    runs-on: ubuntu-latest
    defaults:
      run:
        working-directory: ${{ inputs.working-directory }}
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ inputs.python-version }}
          cache: 'pip'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Lint with ruff
        run: ruff check .

      - name: Run pytest
        run: pytest tests/ --tb=short -q

      - name: Upload coverage
        if: secrets.CODECOV_TOKEN != ''
        run: |
          pip install codecov
          codecov --token ${{ secrets.CODECOV_TOKEN }}
"""

(wf_dir / "lint-and-test.yml").write_text(reusable_wf_yaml)
print("Written: .github/workflows/lint-and-test.yml")

# Inspect the inputs and secrets declared
parsed = yaml.safe_load(reusable_wf_yaml)
wc = parsed["on"]["workflow_call"]
print("\nInputs:")
for name, cfg in wc.get("inputs", {}).items():
    req = "required" if cfg.get("required") else "optional"
    default = f" (default: {cfg['default']!r})" if "default" in cfg else ""
    print(f"  {name}: {cfg['type']} — {req}{default}")
print("\nSecrets:")
for name, cfg in wc.get("secrets", {}).items():
    req = "required" if cfg.get("required") else "optional"
    print(f"  {name}: {req}")


### What just happened?

- We wrote a reusable workflow that exposes **two inputs** (`python-version` required, `working-directory` optional with a default) and **one optional secret** (`CODECOV_TOKEN`).
- The `defaults.run.working-directory` key sets a default working directory for all `run:` steps in that job — cleaner than repeating `cd` in every step.
- **`if: secrets.CODECOV_TOKEN != ''`** gates the upload step — secrets that aren't required are passed as empty strings when not provided, not `null`.
- The Python introspection confirms all declared inputs and their constraints — exactly what you'd use to generate documentation for your shared workflow library.


## Step 2 · Caller workflows — `uses:` at the job level

When calling a reusable workflow, `uses:` appears at the **job level** (not the step level). The entire job is replaced by the called workflow's jobs.

```yaml
jobs:
  call-shared-ci:
    uses: ./.github/workflows/lint-and-test.yml   # same-repo call
    # OR
    uses: org/repo/.github/workflows/lint-and-test.yml@main  # cross-repo call
    with:
      python-version: '3.11'
    secrets:
      CODECOV_TOKEN: ${{ secrets.CODECOV_TOKEN }}
      # OR: secrets: inherit   (passes ALL caller secrets — use with care)
```

Limitations of reusable workflows:
- Max **3 levels** of nesting (a workflow calling a workflow calling a workflow)
- A caller cannot set environment variables that the callee inherits (use inputs instead)
- Reusable workflow jobs always run on their own runner — they are fully isolated


In [ ]:
# Caller 1 — main app CI
caller_app_yaml = """\
name: App CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  lint-and-test:
    uses: ./.github/workflows/lint-and-test.yml
    with:
      python-version: '3.11'
      working-directory: 'app/'
    secrets:
      CODECOV_TOKEN: ${{ secrets.CODECOV_TOKEN }}

  deploy:
    needs: lint-and-test
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to staging
        run: echo "Deploying app to staging..."
"""

# Caller 2 — ML training service CI
caller_ml_yaml = """\
name: ML Service CI

on:
  push:
    paths: ['ml/**']
  pull_request:
    paths: ['ml/**']

jobs:
  lint-and-test:
    uses: ./.github/workflows/lint-and-test.yml
    with:
      python-version: '3.12'
      working-directory: 'ml/'
    secrets: inherit     # passes ALL caller secrets to the reusable workflow
"""

(wf_dir / "app-ci.yml").write_text(caller_app_yaml)
(wf_dir / "ml-ci.yml").write_text(caller_ml_yaml)
print("Written: app-ci.yml and ml-ci.yml")

# Show how each caller maps to the reusable workflow's interface
for name, yaml_text in [("app-ci.yml", caller_app_yaml), ("ml-ci.yml", caller_ml_yaml)]:
    parsed = yaml.safe_load(yaml_text)
    for job_name, job_cfg in parsed["jobs"].items():
        if "uses" in job_cfg:
            print(f"\n{name} → job '{job_name}':")
            print(f"  calls  : {job_cfg['uses']}")
            print(f"  with   : {job_cfg.get('with', {})}")
            secrets = job_cfg.get('secrets', {})
            print(f"  secrets: {secrets if isinstance(secrets, str) else list(secrets.keys())}")


### What just happened?

- **Two callers, one reusable workflow** — both `app-ci.yml` and `ml-ci.yml` call `lint-and-test.yml` with different inputs, eliminating ~30 lines of duplicated YAML each.
- **`secrets: inherit`** (used in `ml-ci.yml`) passes every secret the caller has access to. Convenient, but use with caution in cross-repo calls — prefer explicit secret mapping in public repos.
- **`needs: lint-and-test`** in `app-ci.yml` references the job that calls the reusable workflow — you can chain normal jobs after reusable workflow calls exactly like any other `needs:` dependency.
- **`paths:` filter** in `ml-ci.yml` — this CI only runs when files under `ml/` change, saving runner minutes on unrelated commits.


## Step 3 · Composite actions — shared steps within a job

A **composite action** bundles a sequence of steps into a reusable unit, but it runs *inside* the calling job's runner (not on its own runner).

Anatomy:

```
.github/actions/setup-python-cached/
  action.yml      ← required — defines the action
```

```yaml
# action.yml
name: 'Setup Python with Cache'
description: 'Install Python, restore pip cache, install dependencies'
inputs:
  python-version:
    description: 'Python version'
    required: true
  requirements-file:
    description: 'Path to requirements.txt'
    required: false
    default: 'requirements.txt'
outputs:
  cache-hit:
    description: 'Whether pip cache was restored'
    value: ${{ steps.cache.outputs.cache-hit }}
runs:
  using: 'composite'
  steps:
    - uses: actions/setup-python@v5
      with:
        python-version: ${{ inputs.python-version }}
        cache: 'pip'
    - name: Install dependencies
      shell: bash
      run: pip install -r ${{ inputs.requirements-file }}
```

Key difference from reusable workflows: **composite actions cannot receive secrets** directly. Pass sensitive values via `inputs:` (they will be masked in logs if you mark them `required: true` and don't echo them).


In [ ]:
actions_dir = pathlib.Path(".github/actions/setup-python-cached")
actions_dir.mkdir(parents=True, exist_ok=True)

composite_action_yaml = """\
name: 'Setup Python with Cached Dependencies'
description: 'Install the requested Python version, restore pip cache, and install project dependencies.'

inputs:
  python-version:
    description: 'Python version (e.g. 3.11)'
    required: true
  requirements-file:
    description: 'Path to requirements file'
    required: false
    default: 'requirements.txt'
  cache-suffix:
    description: 'Extra cache key suffix to bust the cache when needed'
    required: false
    default: ''

outputs:
  cache-hit:
    description: 'True if the pip cache was restored from a previous run'
    value: ${{ steps.pip-cache.outputs.cache-hit }}

runs:
  using: composite
  steps:
    - name: Set up Python
      uses: actions/setup-python@v5
      with:
        python-version: ${{ inputs.python-version }}

    - name: Restore pip cache
      id: pip-cache
      uses: actions/cache@v4
      with:
        path: ~/.cache/pip
        key: pip-${{ inputs.python-version }}-${{ hashFiles(inputs.requirements-file) }}-${{ inputs.cache-suffix }}
        restore-keys: |
          pip-${{ inputs.python-version }}-
          pip-

    - name: Install dependencies
      shell: bash
      run: |
        pip install --upgrade pip
        pip install -r ${{ inputs.requirements-file }}
"""

(actions_dir / "action.yml").write_text(composite_action_yaml)
print("Written: .github/actions/setup-python-cached/action.yml")

# Parse and summarise the action interface
parsed = yaml.safe_load(composite_action_yaml)
print(f"\nAction: {parsed['name']}")
print("Inputs:")
for k, v in parsed["inputs"].items():
    req = "required" if v.get("required") else f"optional (default: {v.get('default', '')!r})"
    print(f"  {k}: {req}")
print("Outputs:")
for k, v in parsed["outputs"].items():
    print(f"  {k}: {v['description']}")
print(f"Steps: {len(parsed['runs']['steps'])}")


### What just happened?

- We built a composite action with **3 steps**: Python setup, pip cache restore, and dependency install — all bundled into a reusable unit.
- **`uses: actions/cache@v4`** with `hashFiles(inputs.requirements-file)` creates a deterministic cache key: the cache is only invalidated when `requirements.txt` changes, not on every push.
- The `restore-keys:` fallback chain means a cache miss still gets a partial hit from a different version of the requirements — dramatically faster than a full cold install.
- **`shell: bash`** is **required** on every `run:` step inside a composite action — unlike regular workflow steps, the shell is not inferred automatically.


## Step 4 · Using the composite action in three places within one job

Once the composite action exists, call it with `uses: ./.github/actions/setup-python-cached` from any step in any job. Here we show a workflow that calls it three times for different environments (a common pattern in monorepos with separate service subdirectories).


In [ ]:
composite_caller_yaml = """\
name: Monorepo CI (Composite Action Demo)

on:
  push:
    branches: [main]

jobs:
  build-all:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      # Service A — uses the composite action
      - name: Setup Service A
        uses: ./.github/actions/setup-python-cached
        with:
          python-version: '3.11'
          requirements-file: 'service_a/requirements.txt'
          cache-suffix: 'service-a'

      - name: Test Service A
        working-directory: service_a/
        run: pytest tests/ -q

      # Service B — composite action again, different requirements
      - name: Setup Service B
        uses: ./.github/actions/setup-python-cached
        with:
          python-version: '3.12'
          requirements-file: 'service_b/requirements.txt'
          cache-suffix: 'service-b'

      - name: Test Service B
        working-directory: service_b/
        run: pytest tests/ -q

      # ML pipeline — composite action a third time
      - name: Setup ML pipeline
        uses: ./.github/actions/setup-python-cached
        with:
          python-version: '3.11'
          requirements-file: 'ml/requirements.txt'
          cache-suffix: 'ml'

      - name: Validate ML pipeline
        working-directory: ml/
        run: python validate_pipeline.py --dry-run
"""

(wf_dir / "monorepo-ci.yml").write_text(composite_caller_yaml)
print("Written: .github/workflows/monorepo-ci.yml")

# Count how many times the composite action is called
parsed = yaml.safe_load(composite_caller_yaml)
composite_calls = [
    step for job in parsed["jobs"].values()
    for step in job.get("steps", [])
    if step.get("uses", "").startswith("./.github/actions/setup-python-cached")
]
print(f"\nComposite action called {len(composite_calls)} time(s):")
for call in composite_calls:
    print(f"  - {call['name']}: python-version={call['with']['python-version']}, "
          f"requirements-file={call['with']['requirements-file']}")


### What just happened?

- We called the composite action **three times** in a single job, each with a different `python-version`, `requirements-file`, and `cache-suffix`.
- The `cache-suffix` input lets each service have its own isolated cache key — without it, all three would share a cache and only the first call would get a hit.
- This is the **composite action sweet spot**: repeating a sequence of steps in the same job where runner isolation (a full separate VM) would be overkill.
- Contrast with reusable workflows: if Service A, B, and ML needed to run truly in parallel on separate runners, we'd use a reusable workflow + matrix or three separate job entries with `uses:`.


## Step 5 · Decision tree — reusable workflow vs composite action

| Question | Answer → choose |
|----------|-----------------|
| Do you need a **separate runner** (isolated env, different OS)? | Reusable workflow |
| Do you need to pass **secrets** to the shared code? | Reusable workflow |
| Do you want to run the shared logic **in parallel** alongside other jobs? | Reusable workflow |
| Do you just need to **share a few steps** inside an existing job? | Composite action |
| Do you need the shared steps to access the **calling job's filesystem** without uploads? | Composite action |
| Do you want to publish the action to the **GitHub Marketplace**? | Composite (or JS/Docker) action |

The **golden rule:** reusable workflows give you a full job boundary; composite actions give you reusable steps. Mix them freely — a reusable workflow can contain steps that `uses:` a composite action.


In [ ]:
# Build a decision helper in Python

def recommend_reuse_pattern(
    needs_own_runner: bool = False,
    needs_secrets: bool = False,
    needs_parallel_execution: bool = False,
    sharing_within_same_job: bool = False,
    publish_to_marketplace: bool = False,
) -> str:
    """Return the recommended reuse pattern based on requirements."""
    if needs_own_runner or needs_secrets or needs_parallel_execution:
        reasons = []
        if needs_own_runner:            reasons.append("needs isolated runner")
        if needs_secrets:               reasons.append("needs secrets")
        if needs_parallel_execution:    reasons.append("needs parallel execution")
        return f"→ Reusable workflow  ({', '.join(reasons)})"

    if sharing_within_same_job or publish_to_marketplace:
        reasons = []
        if sharing_within_same_job:     reasons.append("sharing steps within a job")
        if publish_to_marketplace:      reasons.append("publishing to Marketplace")
        return f"→ Composite action   ({', '.join(reasons)})"

    return "→ Either works — prefer composite action for simplicity"


scenarios = [
    {"name": "Deploy job needs AWS_SECRET_ACCESS_KEY",
     "needs_secrets": True},
    {"name": "Lint + test steps shared across 3 services in same job",
     "sharing_within_same_job": True},
    {"name": "Integration tests need their own Ubuntu + Docker setup",
     "needs_own_runner": True},
    {"name": "Cache-warmer steps reused across multiple steps",
     "sharing_within_same_job": True},
    {"name": "Parallel build matrix across 5 configurations",
     "needs_parallel_execution": True},
]

print("Pattern recommendation:")
for s in scenarios:
    name = s.pop("name")
    rec = recommend_reuse_pattern(**s)
    print(f"  [{name}]\n    {rec}\n")


### What just happened?

- We built a **decision function** that encodes the reusable workflow vs composite action choice as explicit boolean flags.
- The function makes the decision tree auditable and testable — great for a team workflow-standards bot.
- **Key insight:** secrets are the clearest forcing function. If the shared code needs a secret, use a reusable workflow — end of discussion.
- The function defaults to composite action for ambiguous cases (both approaches work), following the principle of choosing the simpler option when requirements don't force a choice.


## Step 6 · Outputs from reusable workflows

A reusable workflow can expose outputs that the caller reads via `needs.<job-id>.outputs.<key>`:

```yaml
# In the reusable workflow
on:
  workflow_call:
    outputs:
      image-tag:
        description: 'Docker image tag that was built'
        value: ${{ jobs.build.outputs.image-tag }}

jobs:
  build:
    outputs:
      image-tag: ${{ steps.tag.outputs.value }}
    steps:
      - id: tag
        run: echo "value=sha-${{ github.sha }}" >> $GITHUB_OUTPUT
```

This is the pattern for **build → deploy** pipelines where the deploy job needs the image tag computed by the build job.


In [ ]:
reusable_with_output_yaml = """\
name: Build and Tag (Reusable with Output)

on:
  workflow_call:
    inputs:
      image-name:
        type: string
        required: true
    outputs:
      image-tag:
        description: 'Full image tag including registry'
        value: ${{ jobs.build.outputs.image-tag }}
    secrets:
      REGISTRY_PASSWORD:
        required: true

jobs:
  build:
    runs-on: ubuntu-latest
    outputs:
      image-tag: ${{ steps.compute-tag.outputs.tag }}

    steps:
      - uses: actions/checkout@v4

      - name: Compute image tag
        id: compute-tag
        run: |
          SHORT_SHA=$(echo "${{ github.sha }}" | head -c 8)
          TAG="ghcr.io/${{ github.repository_owner }}/${{ inputs.image-name }}:${SHORT_SHA}"
          echo "tag=${TAG}" >> $GITHUB_OUTPUT
          echo "Computed tag: ${TAG}"

      - name: Build and push Docker image
        run: |
          echo "${{ secrets.REGISTRY_PASSWORD }}" | docker login ghcr.io -u ${{ github.actor }} --password-stdin
          docker build -t ${{ steps.compute-tag.outputs.tag }} .
          docker push ${{ steps.compute-tag.outputs.tag }}
"""

caller_deploy_yaml = """\
name: Build and Deploy

on:
  push:
    branches: [main]

jobs:
  build:
    uses: ./.github/workflows/build-and-tag.yml
    with:
      image-name: 'my-ml-service'
    secrets:
      REGISTRY_PASSWORD: ${{ secrets.REGISTRY_PASSWORD }}

  deploy:
    needs: build
    runs-on: ubuntu-latest
    steps:
      - name: Deploy to Kubernetes
        run: |
          # Consume the image tag output from the build reusable workflow
          IMAGE=${{ needs.build.outputs.image-tag }}
          echo "Deploying ${IMAGE} to prod cluster"
          kubectl set image deployment/my-ml-service app=${IMAGE}
"""

(wf_dir / "build-and-tag.yml").write_text(reusable_with_output_yaml)
(wf_dir / "build-and-deploy.yml").write_text(caller_deploy_yaml)
print("Written: build-and-tag.yml, build-and-deploy.yml")

# Trace the output chain
print("\nOutput propagation chain:")
print("  step 'compute-tag' → $GITHUB_OUTPUT key 'tag'")
print("  job 'build' outputs.image-tag = steps.compute-tag.outputs.tag")
print("  workflow_call outputs.image-tag = jobs.build.outputs.image-tag")
print("  caller: needs.build.outputs.image-tag → deploy step")


### What just happened?

- We traced the **three-level output chain**: step output → job output → workflow output → caller.
- This is the standard pattern for build → deploy pipelines: the build job computes the artifact identifier (image tag, version string, S3 path) and the deploy job consumes it without hardcoding.
- **`head -c 8`** on the 40-char SHA gives a short, human-readable tag — identical to `git rev-parse --short HEAD` but without needing a checkout's git metadata.
- The reusable workflow keeps the secret (`REGISTRY_PASSWORD`) fully internal — the caller doesn't need to know *how* the registry auth works, only that it needs to supply the credential.


## Step 7 · Validation — checking your reuse patterns programmatically

Before pushing, verify that:
1. Every reusable workflow (`workflow_call`) has a matching caller using `uses: ./.github/workflows/<file>`.
2. Every composite action is in `.github/actions/<name>/action.yml`.
3. Callers don't reference inputs or secrets that aren't declared in the callee.


In [ ]:
def audit_reusable_workflows(wf_dir: pathlib.Path) -> dict:
    """Scan .github/workflows/ and return a reuse audit report."""
    callables = {}   # filename → declared inputs/secrets
    callers   = []   # list of (caller_file, job, uses_path, with_keys, secrets_keys)

    for yml_file in sorted(wf_dir.glob("*.yml")):
        try:
            data = yaml.safe_load(yml_file.read_text())
        except yaml.YAMLError:
            continue
        if not isinstance(data, dict) or "jobs" not in data:
            continue

        # Is this a reusable workflow?
        on_block = data.get("on", {})
        if isinstance(on_block, dict) and "workflow_call" in on_block:
            wc = on_block["workflow_call"] or {}
            callables[yml_file.name] = {
                "inputs":  list((wc.get("inputs") or {}).keys()),
                "secrets": list((wc.get("secrets") or {}).keys()),
                "outputs": list((wc.get("outputs") or {}).keys()),
            }

        # Does this workflow call any reusable workflows?
        for job_id, job_cfg in data["jobs"].items():
            if not isinstance(job_cfg, dict):
                continue
            if "uses" in job_cfg:
                uses = job_cfg["uses"]
                callers.append({
                    "caller": yml_file.name,
                    "job":    job_id,
                    "uses":   uses,
                    "with":   list((job_cfg.get("with") or {}).keys()),
                    "secrets_inherit": job_cfg.get("secrets") == "inherit",
                })

    return {"callables": callables, "callers": callers}


report = audit_reusable_workflows(wf_dir)

print("=== Reusable workflows (callables) ===")
for name, info in report["callables"].items():
    print(f"  {name}")
    print(f"    inputs : {info['inputs']}")
    print(f"    secrets: {info['secrets']}")
    print(f"    outputs: {info['outputs']}")

print("\n=== Callers ===")
for c in report["callers"]:
    print(f"  {c['caller']} → job '{c['job']}'")
    print(f"    uses: {c['uses']}")
    print(f"    with: {c['with']}, secrets_inherit: {c['secrets_inherit']}")


### What just happened?

- We built a **workflow auditor** that discovers all reusable workflows and all callers in the `.github/workflows/` directory.
- This is the foundation of a workflow linter: cross-reference `callers[*].with` keys against `callables[uses].inputs` to catch callers passing undeclared inputs.
- **`secrets: inherit`** is flagged separately — it's convenient but bypasses explicit secret declaration, so teams often want to audit it.
- In production you'd also check cross-repo `uses:` references (e.g. `org/shared-actions/.github/workflows/lint.yml@main`) — the same pattern, just with an external path.


In [ ]:
# Challenge: Build the following reuse infrastructure:
#
#   A) A reusable workflow: .github/workflows/python-ci.yml
#      - trigger: workflow_call
#      - input: python-version (string, required)
#      - secret: PYPI_TOKEN (required)
#      - output: package-version (the version extracted from setup.cfg or pyproject.toml)
#      - jobs: one job with steps: checkout, setup-python, run tests, publish to PyPI
#
#   B) Two callers that use python-ci.yml:
#      - stable-release.yml: calls with python-version='3.11'
#      - nightly.yml: calls with python-version='3.12', uses secrets: inherit
#
#   C) Run your audit_reusable_workflows() to verify the structure
#
# Scaffold:

python_ci_yaml = """\
name: Python CI (Reusable)

on:
  workflow_call:
    inputs:
      python-version:
        # TODO: type, required
    secrets:
      PYPI_TOKEN:
        # TODO: required
    outputs:
      package-version:
        description: 'Version string from pyproject.toml'
        # TODO: value pointing to a job output

jobs:
  ci:
    runs-on: ubuntu-latest
    outputs:
      # TODO: expose package-version from a step output
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          # TODO: python-version from input
      - name: Extract version
        id: get-version
        run: |
          # TODO: grep version from pyproject.toml and echo to GITHUB_OUTPUT
      - name: Run tests
        run: pytest tests/ -q
      - name: Publish
        run: |
          # TODO: pip install twine && twine upload using PYPI_TOKEN secret
"""

(wf_dir / "python-ci.yml").write_text(python_ci_yaml)

stable_release_yaml = """\
name: Stable Release
on:
  release:
    types: [published]
jobs:
  release:
    # TODO: uses python-ci.yml, python-version='3.11', explicit PYPI_TOKEN secret
"""
(wf_dir / "stable-release.yml").write_text(stable_release_yaml)

nightly_yaml = """\
name: Nightly
on:
  schedule:
    - cron: '0 3 * * *'
jobs:
  nightly:
    # TODO: uses python-ci.yml, python-version='3.12', secrets: inherit
"""
(wf_dir / "nightly-release.yml").write_text(nightly_yaml)

# Run the auditor to verify your structure
report = audit_reusable_workflows(wf_dir)
print("Callables found:", list(report["callables"].keys()))
print("Callers found:", [(c["caller"], c["uses"]) for c in report["callers"]])
print("\nFill in the TODOs and re-run this cell to see python-ci.yml appear as a callable.")


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| Reusable workflow | Triggered by `workflow_call`; has its own runner; can receive secrets |
| Composite action | Lives in `.github/actions/<name>/action.yml`; shares steps inside a job |
| `secrets: inherit` | Passes all caller secrets to callee — convenient but less explicit |
| `workflow_call` outputs | Step → job → workflow output chain; read by caller via `needs.<id>.outputs.*` |
| `shell: bash` | Required on every `run:` step in a composite action |
| Max nesting depth | Reusable workflows support 3 levels of nesting |
| Pattern decision | Secrets / own runner / parallel → reusable workflow; shared steps in same job → composite action |

> **Tip:** Use reusable workflows when you need a full job with its own runner and environment isolation. Use composite actions when you just need to share a few steps inside a job. The key difference: only reusable workflows can use `secrets: inherit`.

---
## What's next
**Day 4** → Secrets, environments, and protection rules: scoping secrets to environments, requiring manual approval gates, and using OIDC for keyless cloud authentication.

Mark Day 3 complete in your [tracker](../index.html).
